In [18]:

import heapq

# 1. Dynamic Programming
def knapsack_dp(v, w, capacity):
    n = len(v)
    dp = [[0] * (capacity + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        for c in range(capacity + 1):
            dp[i][c] = dp[i - 1][c]
            if w[i - 1] <= c:
                dp[i][c] = max(dp[i][c],
                               v[i - 1] + dp[i - 1][c - w[i - 1]])

    selected = []
    c = capacity

    for i in range(n, 0, -1):
        if dp[i][c] != dp[i - 1][c]:
            selected.append(i)
            c -= w[i - 1]

    return dp[n][capacity], sorted(selected)


# 2. Branch and Bound
def knapsack_bb(v, w, capacity):
    items = sorted(zip(v, w, range(1, len(v) + 1)),
                   key=lambda x: x[0] / x[1], reverse=True)

    def bound(i, profit, weight):
        for val, wt, idx in items[i:]:
            if weight + wt <= capacity:
                weight += wt
                profit += val
            else:
                profit += (capacity - weight) * val / wt
                break
        return profit

    best, selected = 0, []
    pq = [(-bound(0, 0, 0), 0, 0, 0, [])]

    while pq:
        b, i, profit, weight, sel = heapq.heappop(pq)

        if -b <= best or i == len(items):
            continue

        val, wt, idx = items[i]

        # Include or exclude current item
        for p, wgt, s in [(profit + val, weight + wt, sel + [idx]),
                          (profit, weight, sel)]:

            if wgt <= capacity:
                if p > best:
                    best, selected = p, s

                b = bound(i + 1, p, wgt)
                if b > best:
                    heapq.heappush(pq, (-b, i + 1, p, wgt, s))

    return best, sorted(selected)


# Main program
n = int(input("Enter number of items: "))
v = list(map(int, input("Enter values: ").split()))
w = list(map(int, input("Enter weights: ").split()))
c = int(input("Enter capacity: "))

if len(v) != n or len(w) != n or min(w, default=0) <= 0 or c < 0 or any(x < 0 for x in v):
    print("Invalid input")
else:
    profit, items = knapsack_dp(v, w, c)
    print("\nDynamic Programming")
    print("Maximum value:", profit)
    print("Selected items:", items)

    profit, items = knapsack_bb(v, w, c)
    print("\nBranch and Bound")
    print("Maximum value:", profit)
    print("Selected items:", items)


Enter number of items: 3
Enter values: 60 30 50
Enter weights: 20 10 25
Enter capacity: 50

Dynamic Programming
Maximum value: 110
Selected items: [1, 3]

Branch and Bound
Maximum value: 110
Selected items: [1, 3]
